### Processing — Splits MIMO do artigo (São José)

Gera os datasets `data/processed/mimo_dataset_sample_from_papper_{train,test}.parquet`
usando exatamente os mesmos splits de estimação/validação do código
MATLAB original do artigo
[Ferrari, Leandro & Coelho (2025)](../../docs/pappers/Multi-objective%20metaheuristics%20applied%20for%20the%20multivariable%20system%20identification.pdf)
(`data/raw/WRSO and NSWRSO_draft_all/Multi_objective_for_MIMO_systems/MOGWO/main.m`):

- Estimação (treino): linhas `550:1150` (1-indexado, MATLAB) → 601 amostras
- Validação (teste): linhas `1500:2000` (1-indexado, MATLAB) → 501 amostras

Fonte dos dados: `SANEPAR_COMPLETO.txt`, o array que o próprio MATLAB
indexava — usamos ele em vez de `data/raw/mimo_data.xlsx` porque as
contagens de linha não batem entre os dois arquivos (ver
[docs/changes/2026-09-25-reproduzir-mogwo.md](../../docs/changes/2026-09-25-reproduzir-mogwo.md)),
então `SANEPAR_COMPLETO.txt` é a fonte de verdade para reproduzir os
índices exatos do artigo.

Colunas na ordem do artigo: `u1,u2,u3` = frequência das bombas
(`freq_b1..3`), `u4` = vazão de entrada (`vazao_entrada`); `y1,y2,y3` =
vazão de distribuição, nível e pressão do reservatório.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_RAW = Path("../../data/raw")
DATA_PROCESSED = Path("../../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

SANEPAR_PATH = (
    DATA_RAW
    / "WRSO and NSWRSO_draft_all"
    / "Multi_objective_for_MIMO_systems"
    / "MOGWO"
    / "SANEPAR_COMPLETO.txt"
)

COLUMNS = [
    "freq_b1",
    "freq_b2",
    "freq_b3",
    "vazao_entrada",
    "vazao_distribuicao",
    "nivel_res",
    "pressao",
]

# Slices MATLAB 1-indexados inclusivos (main.m) -> slices 0-indexados
TRAIN_SLICE = slice(549, 1150)  # est = 550:1150
TEST_SLICE = slice(1499, 2000)  # val = 1500:2000

In [ ]:
sanepar = np.loadtxt(SANEPAR_PATH)
df = pd.DataFrame(sanepar, columns=COLUMNS)
df.index.name = "sample_idx"

print(df.shape)
df.head()

(2208, 7)


,freq_b1,freq_b2,freq_b3,vazao_entrada,vazao_distribuicao,nivel_res,pressao
sample_idx,,,,,,,
0,0.0,48.500000,0.0,59.161747,41.319435,2.313008,16.42
1,0.0,48.500000,0.0,59.349953,36.163658,2.376330,16.42
2,0.0,48.500000,0.0,38.935471,33.399830,2.433640,16.42
3,0.0,45.197014,0.0,36.185631,32.003647,2.449207,16.00
4,0.0,50.150032,0.0,36.003067,32.846390,2.458931,16.00


### Splits de estimação (treino) e validação (teste)

In [3]:
train = df.iloc[TRAIN_SLICE].reset_index()
test = df.iloc[TEST_SLICE].reset_index()

print("treino:", train.shape)
print("teste:", test.shape)

assert train.shape == (601, len(COLUMNS) + 1)
assert test.shape == (501, len(COLUMNS) + 1)

treino: (601, 8)
teste: (501, 8)


### Salvar em `data/processed/`

In [4]:
train_path = DATA_PROCESSED / "mimo_dataset_sample_from_papper_train.parquet"
test_path = DATA_PROCESSED / "mimo_dataset_sample_from_papper_test.parquet"

train.to_parquet(train_path)
test.to_parquet(test_path)

print("salvo:", train_path)
print("salvo:", test_path)

salvo: ../../data/processed/mimo_dataset_sample_from_papper_train.parquet
salvo: ../../data/processed/mimo_dataset_sample_from_papper_test.parquet
